# Backtesting + ELB-censored scoring\n\nMany macro forecasting papers score short-term interest rates using an **ELB-censored convention**:\n\n- **Realized values** are floored at an ELB (e.g., 0.0 or 0.25).\n- Optionally, **forecast draws** are also floored before computing scores.\n\nThis notebook runs a tiny rolling backtest and shows how to apply those evaluation-time\ntransformations consistently via `evaluation.elb_censor` (the same keys used by the CLI).\n

In [ ]:
import numpy as np\nimport pandas as pd\n\nfrom srvar import Dataset\nfrom srvar.api import fit, forecast\nfrom srvar.config import build_evaluation_config\nfrom srvar.evaluation import compute_metrics_rows, prepare_evaluation_inputs\nfrom srvar.spec import ModelSpec, PriorSpec, SamplerConfig\n\ndf = pd.read_csv("../data/example.csv")\ndf["date"] = pd.to_datetime(df["date"])\n\nvalues = df[["r", "y"]].to_numpy(dtype=float)\nds = Dataset.from_arrays(values=values, variables=["r", "y"], time_index=df["date"])\n\nds.T\n

## Mini backtest\n\nTo keep runtime low, we:\n\n- fit a simple homoskedastic NIW BVAR (no ELB constraint in the model)\n- use a small number of origins\n- forecast horizons 1–2\n\nThen we compare metrics under different `evaluation.elb_censor` settings.\n

In [ ]:
p = 2\nhorizons = [1, 2]\nmax_h = max(horizons)\n\nmodel = ModelSpec(p=p, include_intercept=True)\nprior = PriorSpec.niw_default(k=1 + ds.N * model.p, n=ds.N)\nsampler = SamplerConfig(draws=200, burn_in=0, thin=1)\n\norigins = list(range(40, 48))\npred_draws = 200\n\nrng = np.random.default_rng(0)\n\nforecasts = []\ny_true = []\nfor origin_end in origins:\n    train_ds = Dataset.from_arrays(\n        values=ds.values[: origin_end + 1, :],\n        variables=ds.variables,\n        time_index=ds.time_index[: origin_end + 1],\n    )\n\n    fit_res = fit(train_ds, model, prior, sampler, rng=rng)\n    fc = forecast(fit_res, horizons=horizons, draws=pred_draws, rng=rng)\n\n    forecasts.append(fc)\n    y_true.append(ds.values[origin_end + 1 : origin_end + 1 + max_h, :])\n\ny_true = np.stack(y_true)\ny_true.shape\n

## Metrics without ELB censoring\n\nThe evaluation config is a plain Python dictionary, but it uses the same keys you would\nset in YAML under `evaluation:`.\n

In [ ]:
cfg_no_censor = {\n    "evaluation": {\n        "coverage": {"enabled": False},\n        "crps": {"enabled": True, "use_latent": False},\n        "elb_censor": {"enabled": False},\n        "metrics_table": True,\n    }\n}\n\nev_no_censor = build_evaluation_config(cfg_no_censor, variables=list(ds.variables), horizons=horizons)\ny_true_eval, forecasts_eval = prepare_evaluation_inputs(\n    y_true=y_true, forecasts=forecasts, variables=list(ds.variables), evaluation=ev_no_censor\n)\n\nrows = compute_metrics_rows(\n    forecasts=forecasts_eval,\n    y_true=y_true_eval,\n    variables=list(ds.variables),\n    evaluation=ev_no_censor,\n)\n\ndf_no = pd.DataFrame(rows)\ndf_no[df_no["variable"] == "r"]\n

## ELB-censored scoring (realized only)\n\nHere we floor the realized interest rate at `bound=0.0`, but we leave forecast draws\nunchanged.\n

In [ ]:
cfg_realized_only = {\n    "evaluation": {\n        "coverage": {"enabled": False},\n        "crps": {"enabled": True, "use_latent": False},\n        "elb_censor": {\n            "enabled": True,\n            "bound": 0.0,\n            "variables": ["r"],\n            "censor_realized": True,\n            "censor_forecasts": False,\n        },\n        "metrics_table": True,\n    }\n}\n\nev_realized_only = build_evaluation_config(\n    cfg_realized_only, variables=list(ds.variables), horizons=horizons\n)\ny_true_eval, forecasts_eval = prepare_evaluation_inputs(\n    y_true=y_true, forecasts=forecasts, variables=list(ds.variables), evaluation=ev_realized_only\n)\n\nrows = compute_metrics_rows(\n    forecasts=forecasts_eval,\n    y_true=y_true_eval,\n    variables=list(ds.variables),\n    evaluation=ev_realized_only,\n)\n\ndf_realized = pd.DataFrame(rows)\ndf_realized[df_realized["variable"] == "r"]\n

## ELB-censored scoring (realized + forecasts)\n\nThis is the common “interest-rate scoring” convention when comparing constrained (ELB)\nand unconstrained models: apply the same ELB floor to forecast draws at evaluation time.\n

In [ ]:
cfg_realized_and_fc = {\n    "evaluation": {\n        "coverage": {"enabled": False},\n        "crps": {"enabled": True, "use_latent": False},\n        "elb_censor": {\n            "enabled": True,\n            "bound": 0.0,\n            "variables": ["r"],\n            "censor_realized": True,\n            "censor_forecasts": True,\n        },\n        "metrics_table": True,\n    }\n}\n\nev_realized_and_fc = build_evaluation_config(\n    cfg_realized_and_fc, variables=list(ds.variables), horizons=horizons\n)\ny_true_eval, forecasts_eval = prepare_evaluation_inputs(\n    y_true=y_true, forecasts=forecasts, variables=list(ds.variables), evaluation=ev_realized_and_fc\n)\n\nrows = compute_metrics_rows(\n    forecasts=forecasts_eval,\n    y_true=y_true_eval,\n    variables=list(ds.variables),\n    evaluation=ev_realized_and_fc,\n)\n\ndf_both = pd.DataFrame(rows)\ndf_both[df_both["variable"] == "r"]\n

## Compare RMSE / CRPS across conventions\n\nBelow is a compact comparison table for the interest rate variable.\n\nNotes:\n- `compute_metrics_rows(...)` outputs horizons `1..max(horizons)` (full grid).\n- To score the **latent shadow rate** (ELB models), set `evaluation.crps.use_latent: true`.\n

In [ ]:
def _summarize(label: str, df: pd.DataFrame) -> pd.DataFrame:\n    out = df.loc[df["variable"] == "r", ["horizon", "rmse", "mae", "crps"]].copy()\n    out.insert(0, "variant", label)\n    return out\n\n\nsummary = pd.concat(\n    [\n        _summarize("none", df_no),\n        _summarize("realized_only", df_realized),\n        _summarize("realized_and_forecasts", df_both),\n    ],\n    ignore_index=True,\n)\n\nsummary\n

## What to tweak\n\n- Increase `pred_draws` to stabilize CRPS estimates.\n- Increase `len(origins)` for a more realistic backtest.\n- Change `bound` to match your ELB convention (e.g., 0.25 for quarterly averages).\n